In [2]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import matplotlib.pyplot as plt
import numpy as np

In [26]:
cluster = PBSCluster(
    cores=1,
    memory='40GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.174:40699,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# data i

In [13]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
hi_ds = xr.open_dataset(path+'hourly_HI.nc')
hi_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 30GB ...

In [14]:
rh_ds = xr.open_dataset(path+'ERA5_RH.nc')
rh_ds = rh_ds['__xarray_dataarray_variable__']
rh_ds.rename('RH')

<xarray.DataArray 'RH' (time: 755304, latitude: 82, longitude: 121)> Size: 30GB
[7494126288 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [15]:
t_ds = xr.open_zarr(path+'sfc_hourly_temp')
t_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2T     (time, latitude, longitude) float32 30GB dask.array<chunksize=(22888, 82, 121), meta=np.ndarray>

# get max hi time stamps

In [16]:
# chunk for dask
hi_ds = hi_ds.chunk(time=17166)
t_ds = t_ds['VAR_2T'].chunk(time=17166)
rh_ds = rh_ds.chunk(time=17166)
t_ds

<xarray.DataArray 'VAR_2T' (time: 755304, latitude: 82, longitude: 121)> Size: 30GB
dask.array<rechunk-merge, shape=(755304, 82, 121), dtype=float32, chunksize=(17166, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [17]:
# make hour dim lazily
hi_ds_coarse = hi_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
t_ds_coarse = t_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
rh_ds_coarse = rh_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
t_ds_coarse

<xarray.DataArray 'VAR_2T' (day: 31471, hour: 24, latitude: 82, longitude: 121)> Size: 30GB
dask.array<reshape, shape=(31471, 24, 82, 121), dtype=float32, chunksize=(715, 24, 82, 121), chunktype=numpy.ndarray>
Coordinates:
    time       (day, hour) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day, hour

In [18]:
hidmax_idx = hi_ds_coarse['HI_hourly'].argmax(dim='hour')
hidmax_idx

<xarray.DataArray 'HI_hourly' (day: 31471, latitude: 82, longitude: 121)> Size: 2GB
dask.array<nanarg_agg-aggregate, shape=(31471, 82, 121), dtype=int64, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [20]:
def sel_at_idx(ds, idx_ds):
    # add hour dimension to indexing array
    idx_ds = np.expand_dims(idx_ds, axis=-1)
    # select along hour axis and remove hour axis
    return np.take_along_axis(ds, idx_ds, axis=-1).squeeze(axis=-1)

In [21]:
t_peakHI = xr.apply_ufunc(sel_at_idx, t_ds_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[t_ds_coarse.dtype]
                             )
t_peakHI = t_peakHI.rename('T_during_HIdmax')
t_peakHI

<xarray.DataArray 'T_during_HIdmax' (day: 31471, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31471, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [23]:
rh_peakHI = xr.apply_ufunc(sel_at_idx, rh_ds_coarse, hidmax_idx,
                               input_core_dims=[['hour'], []],
                               output_core_dims=[[]],
                               dask='parallelized',
                               output_dtypes=[rh_ds_coarse.dtype]
                              )
rh_peakHI = rh_peakHI.rename('RH_during_HIdmax')
rh_peakHI

<xarray.DataArray 'RH_during_HIdmax' (day: 31471, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31471, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [24]:
timestamps = hi_ds['time'].isel(time=slice(0, None, 24)).dt.floor("1D")
timestamps

<xarray.DataArray 'floor' (time: 31471)> Size: 252kB
array(['1940-01-01T00:00:00.000000000', '1940-01-02T00:00:00.000000000',
       '1940-01-03T00:00:00.000000000', ...,
       '2026-02-26T00:00:00.000000000', '2026-02-27T00:00:00.000000000',
       '2026-02-28T00:00:00.000000000'],
      shape=(31471,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 252kB 1940-01-01 1940-01-02 ... 2026-02-28

In [25]:
t_peakHI = t_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop('day')
rh_peakHI = rh_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop('day')
rh_peakHI

/glade/derecho/scratch/acruz/tmp/ipykernel_107288/2343685459.py:1: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  t_peakHI = t_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop('day')
/glade/derecho/scratch/acruz/tmp/ipykernel_107288/2343685459.py:2: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  rh_peakHI = rh_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop('day')


<xarray.DataArray 'RH_during_HIdmax' (time: 31471, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31471, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 252kB 1940-01-01 1940-01-02 ... 2026-02-28
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [27]:
t_peakHI.to_netcdf(path+'T_during_peak_HI.nc', mode='w')

In [28]:
rh_peakHI.to_netcdf(path+'RH_during_peak_HI.nc')

In [29]:
client.shutdown()